# Model: Random Forest

Owner: **Daniel**

## Imports

These are just some example ones, please feel free to remove or add any

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

MODEL_NAME = "random_forest"

## Load preprocessed data

Please load the same preprocessed train and test files.

Do NOT re-clean or re-derive features here. If something looks wrong, or you want to add another feature, please let the team know and we can add it to the csv that we will all use.

We want all the models to have the same data to have a fair comparison.

In [2]:
DATA_DIR = Path("../../data/NSW")
RESULTS_DIR = Path("../../results")
RESULTS_DIR.mkdir(exist_ok=True)

# TODO: swap this for the shared train.csv / test.csv once the team agrees on one.
# For now, split the cleaned dataset here ourselves: chronological 80/20 split,
# i.e. the most recent ~20% of rows (~2.2 years) are held out as the test set.
df = pd.read_csv(DATA_DIR / "nsw_cleaned.csv", parse_dates=["DATETIME"]).set_index("DATETIME").sort_index()

split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

print(f"train: {train.index.min()} -> {train.index.max()}  ({len(train)} rows)")
print(f"test:  {test.index.min()} -> {test.index.max()}  ({len(test)} rows)")

train: 2010-01-01 00:00:00 -> 2018-12-20 04:30:00  (157210 rows)
test:  2018-12-20 05:00:00 -> 2021-03-18 00:00:00  (39303 rows)


## Features and target

You're free to use a subset of these, or engineer new features from them (e.g. lags, rolling averages, calendar features from `DATETIME`) — as the previous section also said, just don't add any additional raw features.

If you think something's missing, mention it to the team so everyone can decide whether to add it to the shared file.

In [3]:
# Temperature-only for now, per team decision - revisit once other features are validated.
TARGET = "TOTALDEMAND"
FEATURES = ["TEMPERATURE"]

## Training the model

In this section please train and implement the model you have been assigned. There are lots of available packages, i.e. sklearn, that should be easily implementable. The training model should only need to be a few lines of code.

In [4]:
# Base estimator; hyperparameters are chosen via the time-series cross-validation below.
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

In [5]:
# Cross-validate on the training data only, using expanding-window time-series splits
# so we never validate on data that precedes what the model was fit on.
tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 5],
}

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
grid_search.fit(train[FEATURES], train[TARGET])

cv_results = (
    pd.DataFrame(grid_search.cv_results_)
    .sort_values("rank_test_score")[["params", "mean_test_score", "std_test_score", "rank_test_score"]]
)
cv_results["mean_test_rmse"] = -cv_results["mean_test_score"]

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV RMSE: {-grid_search.best_score_:.2f} MW")

# GridSearchCV refits this on the full training set automatically (refit=True by default).
model = grid_search.best_estimator_
cv_results.head(10)

Best params: {'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 300}
Best CV RMSE: 1218.96 MW


,params,mean_test_score,std_test_score,rank_test_score,mean_test_rmse
7,"{'max_depth': 10, 'min_samples_leaf': 5, 'n_es...",-1218.961035,52.013744,1,1218.961035
6,"{'max_depth': 10, 'min_samples_leaf': 5, 'n_es...",-1219.030037,52.157210,2,1219.030037
5,"{'max_depth': 10, 'min_samples_leaf': 1, 'n_es...",-1219.187497,52.117714,3,1219.187497
4,"{'max_depth': 10, 'min_samples_leaf': 1, 'n_es...",-1219.244882,52.255638,4,1219.244882
11,"{'max_depth': 20, 'min_samples_leaf': 5, 'n_es...",-1220.065905,52.700358,5,1220.065905
3,"{'max_depth': None, 'min_samples_leaf': 5, 'n_...",-1220.068228,52.702838,6,1220.068228
10,"{'max_depth': 20, 'min_samples_leaf': 5, 'n_es...",-1220.129701,52.834729,7,1220.129701
2,"{'max_depth': None, 'min_samples_leaf': 5, 'n_...",-1220.133573,52.838340,8,1220.133573
9,"{'max_depth': 20, 'min_samples_leaf': 1, 'n_es...",-1220.511180,52.892028,9,1220.511180
1,"{'max_depth': None, 'min_samples_leaf': 1, 'n_...",-1220.513764,52.894692,10,1220.513764


## Validating the model

Produce a `predictions` array or Series covering the full test period, aligned to `test.index`.

Do NOT use this step to tune your model, repeatedly checking against the test set and adjusting based on it will cause you to overfit.

In [6]:
predictions = model.predict(test[FEATURES])

## Validation checklist

Every model notebook uses this same function, so the numbers are computed the same way for everyone:
- **RMSE**: root mean squared error, in MW
- **MAE**: mean absolute error, in MW
- **MAPE**: mean absolute percentage error, in %

In [7]:
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    metrics = {"model": model_name, "rmse": rmse, "mae": mae, "mape_pct": mape}
    print(metrics)
    return metrics


metrics = evaluate(test[TARGET], predictions, MODEL_NAME)

{'model': 'random_forest', 'rmse': np.float64(1216.320913523045), 'mae': 985.2558874276748, 'mape_pct': 13.36018410599362}


## Save results

This writes your predictions and metrics into the shared `results/` folder so they can be pulled together for the report. Don't change the file paths, `MODEL_NAME`, or column names below.

In [8]:
pd.Series(predictions, index=test.index, name=MODEL_NAME).to_csv(RESULTS_DIR / f"{MODEL_NAME}_predictions.csv")

comparison_path = RESULTS_DIR / "model_comparison.csv"
this_run = pd.DataFrame([metrics])

if comparison_path.exists():
    existing = pd.read_csv(comparison_path)
    existing = existing[existing["model"] != MODEL_NAME]
    this_run = pd.concat([existing, this_run], ignore_index=True)

this_run.to_csv(comparison_path, index=False)
this_run

,model,rmse,mae,mape_pct
0,random_forest,1216.320914,985.255887,13.360184
